# Working with requests Library

The requests library helps your Python code talk to websites using HTTP. It allows you to send HTTP requests (like visiting a URL), and receive HTML responses (like viewing a webpage's source code).

## Sending a Basic GET Request

In [ ]:
import requests

url = "https://quotes.toscrape.com"
response = requests.get(url)

In [ ]:
response.status_code  # 200 means success

In [ ]:
if response.status_code == 200:
    print("Page fetched successfully!")
else:
    print("Something went wrong:", response.status_code)


In [ ]:
response.text         # HTML content of the page

response.text[:1000] # display only 1000 chracter

## Setting Headers (like User-Agent)
Some websites block bots. You can send headers to pretend to be a browser.

In [ ]:
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
}

response = requests.get(url, headers=headers)
print(response.text)


## Sending Query Parameters
You can pass extra information in the URL using parameters.

In [ ]:
params = {
    "search": "laptop",
    "price": "under1000"
}

response = requests.get("https://example.com/products", params=params)
print(response.url)  # See the full URL with parameters


# https://example.com/products?search=laptop&price=under1000

## Handling Timeouts and Errors
Always set a timeout so your program doesn’t hang.

In [ ]:
try:
    response = requests.get("https://example.com", timeout=5)
    print(response.status_code)
except requests.exceptions.Timeout:
    print("Request timed out!")
except requests.exceptions.RequestException as e:
    print("An error occurred:", e)


## Using requests.Session() for Multiple Requests
If you're scraping multiple pages from the same site, use a Session object.

In [ ]:
session = requests.Session()
session.headers.update({"User-Agent": "Mozilla/5.0"})

response1 = session.get("https://example.com/page1")
response2 = session.get("https://example.com/page2")


# Basics of HTML & the DOM

DOM = Document Object Model

- It’s like a tree structure of HTML elements.

Each element is a node that can have:
- A tag name
- Attributes (e.g., class, id)
- Children (nested elements)

In [ ]:
<div class="quote">
  <span class="text">“The world is full of quotes.”</span>
  <span>
    <small class="author">John Doe</small>
  </span>
</div>


In this:
- div is a container with class quote
- Inside it is a span with class text → contains the quote
- Another span contains a small tag with class author → contains the author’s name

## Visualizing the DOM
Here’s how that quote HTML looks in a tree (DOM):

In [ ]:
div (class="quote")
├── span (class="text") → “The world is full of quotes.”
└── span
    └── small (class="author") → John Doe


You’ll use this structure with BeautifulSoup to find and extract the elements you need.

# Introduction to BeautifulSoup
BeautifulSoup is a Python library used to parse HTML and extract data from web pages. It makes it easy to navigate the DOM tree and find specific elements.

In [ ]:
import requests
from bs4 import BeautifulSoup

url = "https://quotes.toscrape.com"
response = requests.get(url)
soup = BeautifulSoup(response.text, "lxml")  # or "html.parser" or "html5lib"

# response.text is html

In [2]:
from bs4 import BeautifulSoup

html = """
<html>
  <head>
    <title>My First Scraper</title>
  </head>
  <body>
    <h1>Hello Web Scraping!</h1>
  </body>
</html>
"""

soup = BeautifulSoup(html, "lxml")
soup

<html>
<head>
<title>My First Scraper</title>
</head>
<body>
<h1>Hello Web Scraping!</h1>
</body>
</html>

In [5]:
print(soup.title)         # <title>My First Scraper</title>

<title>My First Scraper</title>


In [6]:
soup.title.text    # My First Scraper

'My First Scraper'

In [7]:
soup.title.string    # My First Scraper

'My First Scraper'

In [8]:
soup.h1

<h1>Hello Web Scraping!</h1>

In [11]:
soup.h1.text

'Hello Web Scraping!'

In [9]:
soup.body

<body>
<h1>Hello Web Scraping!</h1>
</body>

In [10]:
soup.body.text

'\nHello Web Scraping!\n'

## Common BeautifulSoup Methods

In [21]:
from bs4 import BeautifulSoup

html = """
<html>
  <head>
    <title>My Website</title>
  </head>
  <body>
    <h1>Welcome to My Website</h1>
    <p class="intro">This is the intro paragraph.</p>
    <a href="https://example.com/page1">Page 1</a>
    <a href="https://example.com/page2" class="external">Page 2</a>
    <a href="/contact">Contact</a>
  </body>
</html>
"""

soup = BeautifulSoup(html, "lxml")


### find()
`find()` – Finds the first matching element

In [22]:
first_link = soup.find("a") # Gets the first matching <a> tag
first_link

<a href="https://example.com/page1">Page 1</a>

In [23]:
first_link.text

'Page 1'

In [24]:
p_tag = soup.find("p", {'class':'intro'})
p_tag

<p class="intro">This is the intro paragraph.</p>

In [25]:
p_tag.text

'This is the intro paragraph.'

In [26]:
# same as above: just another syntax
p_tag = soup.find("p", class_='intro')
p_tag.text

'This is the intro paragraph.'

### find_all()

`find_all()` – Finds all matching elements (returns a list)

In [27]:
all_a_tags = soup.find_all("a")
all_a_tags

[<a href="https://example.com/page1">Page 1</a>,
 <a class="external" href="https://example.com/page2">Page 2</a>,
 <a href="/contact">Contact</a>]

In [28]:
all_a_tags = soup.find_all("a")

for a_tag in all_a_tags:
    print(a_tag.text)

Page 1
Page 2
Contact


### select()

`select()` – Uses CSS selectors to find elements (more powerful). Returns a list.

In [29]:
external_links = soup.select("a.external")  # select <a> tags with class="external"
print(external_links[0].get("href"))        # Output: https://example.com/page2

https://example.com/page2


In [30]:
intro_para = soup.select("p.intro")
print(intro_para[0].text)  # Output: This is the intro paragraph.

This is the intro paragraph.


In [16]:
html = """
    <div class="quote">
        <span class="text">“The world as we have created it is a process of our thinking.”</span>
        <span>
            <small class="author">Albert Einstein</small>
        </span>
    </div>
"""

soup = BeautifulSoup(html, "lxml")

In [18]:
# find all <span> elements with class='text' inside a <div> element with class='quote'
quotes = soup.select("div.quote span.text")

for q in quotes:
    print(q.text)


“The world as we have created it is a process of our thinking.”


In [1]:
from bs4 import BeautifulSoup

html = """
<p class="intro highlighted special">Welcome!</p>
"""

soup = BeautifulSoup(html, "lxml")

In [2]:
soup.select("p.intro.highlighted")

[<p class="intro highlighted special">Welcome!</p>]

In [3]:
soup.select("p.intro.highlighted.special")

[<p class="intro highlighted special">Welcome!</p>]

In [4]:
soup.find_all("p", class_=["intro", "highlighted"])

# NOTE: This finds tags that have either class, not necessarily both

[<p class="intro highlighted special">Welcome!</p>]

In [5]:
from bs4 import BeautifulSoup

html = """
<html>
  <body>
    <p class="intro highlighted">First paragraph</p>
    <p class="intro">Second paragraph</p>
    <p class="highlighted">Third paragraph</p>
  </body>
</html>
"""

soup = BeautifulSoup(html, "lxml")

# CSS selector: must have BOTH intro AND highlighted
results = soup.select("p.intro.highlighted")
for p in results:
    print(p.text)


First paragraph


### select_one()
Used to get only the first element that matches a CSS selector.

In [61]:
from bs4 import BeautifulSoup

html = """
<html>
  <body>
    <div class="quote">
      <span class="text">“Be yourself; everyone else is already taken.”</span>
      <span class="author">Oscar Wilde</span>
    </div>
    <div class="quote">
      <span class="text">“So many books, so little time.”</span>
      <span class="author">Frank Zappa</span>
    </div>
  </body>
</html>
"""

soup = BeautifulSoup(html, "lxml")

In [62]:
# Get the first quote div
first_quote = soup.select_one("div.quote")
first_quote

<div class="quote">
<span class="text">“Be yourself; everyone else is already taken.”</span>
<span class="author">Oscar Wilde</span>
</div>

In [63]:
# Get just the first quote text
first_text = soup.select_one("div.quote span.text").text
print("Quote:", first_text)

Quote: “Be yourself; everyone else is already taken.”


In [64]:
# Get the first author
first_author = soup.select_one("div.quote span.author").text
print("Author:", first_author)

Author: Oscar Wilde


### get()

`get()` – Get the value of an attribute (like `href`, `class`, etc.)

In [65]:
from bs4 import BeautifulSoup

html = """
<html>
  <head>
    <title>My Website</title>
  </head>
  <body>
    <h1>Welcome to My Website</h1>
    <p class="intro">This is the intro paragraph.</p>
    <a href="https://example.com/page1">Page 1</a>
    <a href="https://example.com/page2" class="external">Page 2</a>
    <a href="/contact">Contact</a>
  </body>
</html>
"""

soup = BeautifulSoup(html, "lxml")


In [66]:
link = soup.find("a")

link.get("href")  # Get URL from <a href="">

'https://example.com/page1'

In [67]:
links = soup.find_all("a")

for link in links:
    print(link.get("class"))

None
['external']
None


**Example – Scrape Quotes and Authors**

In [68]:
import requests
from bs4 import BeautifulSoup

url = "https://quotes.toscrape.com"
response = requests.get(url)
soup = BeautifulSoup(response.text, "lxml")  # or "html.parser"

quotes = soup.find_all("div", class_="quote")

for quote in quotes:
    text = quote.find("span", class_="text").text
    author = quote.find("small", class_="author").text
    print(f"{text} — {author}")


“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.” — Albert Einstein
“It is our choices, Harry, that show what we truly are, far more than our abilities.” — J.K. Rowling
“There are only two ways to live your life. One is as though nothing is a miracle. The other is as though everything is a miracle.” — Albert Einstein
“The person, be it gentleman or lady, who has not pleasure in a good novel, must be intolerably stupid.” — Jane Austen
“Imperfection is beauty, madness is genius and it's better to be absolutely ridiculous than absolutely boring.” — Marilyn Monroe
“Try not to become a man of success. Rather become a man of value.” — Albert Einstein
“It is better to be hated for what you are than to be loved for what you are not.” — André Gide
“I have not failed. I've just found 10,000 ways that won't work.” — Thomas A. Edison
“A woman is like a tea bag; you never know how strong it is until it's in hot water.” — Eleanor Roos

| Method          | Purpose                          |
| --------------- | -------------------------------- |
| `.text`         | Get the text inside a tag        |
| `.get("href")`  | Get an attribute value           |
| `.find()`       | Find one child element           |
| `.find_all()`   | Find all matching children       |
| `.parent`       | Go to parent element             |
| `.children`     | Iterate over direct children     |
| `.descendants`  | Iterate over all nested elements |
| `.next_sibling` | Next element at the same level   |


## DOM navigation features

DOM navigation features in BeautifulSoup that let you move around the HTML structure like a tree:

In [7]:
from bs4 import BeautifulSoup

html = """
<html>
  <body>
    <div class="container">
      <h1>Main Heading</h1>
      <p class="intro">Intro paragraph</p>
      <p class="details">More details</p>
    </div>
  </body>
</html>
"""

soup = BeautifulSoup(html, "lxml")


### .parent

Finds the immediate parent of a tag

In [9]:
p_tag = soup.find("p", class_="intro")

p_tag.parent

<div class="container">
<h1>Main Heading</h1>
<p class="intro">Intro paragraph</p>
<p class="details">More details</p>
</div>

In [10]:
p_tag.parent.name

'div'

### .children

Gives a generator of a tag’s direct children (only first level down)

In [11]:
div_tag = soup.find("div", class_="container")
div_tag.children

In [12]:
for child in div_tag.children:
    print(child.name)

None
h1
None
p
None
p
None


These are the direct child tags inside the `<div>`.

In [13]:
div_tag = soup.find("div", class_="container")
for child in div_tag.children:
    print(child.name)


None
h1
None
p
None
p
None


### .descendants
All children at any level (deep nested)

In [14]:
div_tag.descendants

<generator object Tag.descendants at 0x000001F707338740>

In [15]:
for desc in div_tag.descendants:
    if desc.name is not None:
        print(desc.name)


h1
p
p


In this simple example `.children` and `.descendants` give the same output, but in deeply nested HTML, descendants goes through every level.

### .next_sibling / .previous_sibling
Move between sibling elements on the same level

In [16]:
p_intro = soup.find("p", class_="intro")

p_intro.next_sibling         # Often a '\n' (whitespace)

'\n'

In [17]:
p_intro.next_sibling.next_sibling  # Actual next element


<p class="details">More details</p>

n real HTML, elements often have whitespace `'\n'` between them, so you might need `.next_sibling.next_sibling`.

To get the previous sibling:

In [18]:
p_intro.previous_sibling         # '\n'


'\n'

In [19]:
p_intro.previous_sibling.previous_sibling  # <h1>Main Heading</h1>


<h1>Main Heading</h1>

### .contents
It returns a list of the tag's direct children (as Tag or NavigableString objects).

Think of it as getting all the elements that are directly inside a tag — but not nested deeper.

**Note:** It includes whitespace (like \n) as separate items.

In [69]:
from bs4 import BeautifulSoup

html = """
<div class="quote">
  <span class="text">“Be yourself; everyone else is already taken.”</span>
  <span class="author">Oscar Wilde</span>
</div>
"""

soup = BeautifulSoup(html, "lxml")

div = soup.find("div", class_="quote")
div.contents

['\n',
 <span class="text">“Be yourself; everyone else is already taken.”</span>,
 '\n',
 <span class="author">Oscar Wilde</span>,
 '\n']

In [70]:
children = div.contents

for i, child in enumerate(children):
    print(f"Child {i}:", child)


Child 0: 

Child 1: <span class="text">“Be yourself; everyone else is already taken.”</span>
Child 2: 

Child 3: <span class="author">Oscar Wilde</span>
Child 4: 



In [71]:
for child in div.contents:
    if child.name:  # ignore newlines and strings
        print(child.text)


“Be yourself; everyone else is already taken.”
Oscar Wilde


## Using CSS Selectors with select()

In [56]:
from bs4 import BeautifulSoup

html = """
<div class="quote">
  <span class="text">“Be yourself; everyone else is already taken.”</span>
  <span class="author">Oscar Wilde</span>
</div>

<div class="quote">
  <span class="text">“So many books, so little time.”</span>
  <span class="author">Frank Zappa</span>
</div>
"""

soup = BeautifulSoup(html, "lxml")


In [57]:
# Select all links
links = soup.select("a")
links

[]

In [60]:
# Select by class
quotes = soup.select("div.quote")
quotes

[<div class="quote">
 <span class="text">“Be yourself; everyone else is already taken.”</span>
 <span class="author">Oscar Wilde</span>
 </div>,
 <div class="quote">
 <span class="text">“So many books, so little time.”</span>
 <span class="author">Frank Zappa</span>
 </div>]

In [59]:
# Loop through each quote block and extract text
for quote in quotes:
    quote_text = quote.select_one("span.text").text
    author = quote.select_one("span.author").text
    print(f"{quote_text} - {author}")

“Be yourself; everyone else is already taken.” - Oscar Wilde
“So many books, so little time.” - Frank Zappa


In [53]:
# Select specific span inside div
texts = soup.select("div.quote span.text")
texts

[<span class="text">“Be yourself; everyone else is already taken.”</span>,
 <span class="text">“So many books, so little time.”</span>]

In [54]:
for t in texts:
    print(t.text)

“Be yourself; everyone else is already taken.”
“So many books, so little time.”


# Extracting Data from HTML Tags
Now that you know how to use BeautifulSoup to parse and search HTML, let’s extract different types of data from real web pages: text, links, images, lists, and tables.


## Extracting Text
You usually want the content inside a tag, not the tag itself.

In [ ]:
from bs4 import BeautifulSoup

html = """
<html>
  <body>
    <div class="container">
      <h1>Main Heading</h1>
      <p class="intro">Intro paragraph</p>
      <p class="details">More details</p>
    </div>
  </body>
</html>
"""

soup = BeautifulSoup(html, "lxml")


In [23]:
p_tags = soup.find_all('p')

for p_tag in p_tags:
    print(p_tag.text)  # or tag.get_text()

Intro paragraph
More details


## Extracting Attributes (e.g., href, src, alt)
HTML tags often have attributes like `href`, `src`, `alt`, `id`, or `class`.

### Example: Extracting links
`<a href="https://example.com">Visit</a>`

In [24]:
from bs4 import BeautifulSoup

html = """
<a href="https://example.com">Visit</a>
"""

soup = BeautifulSoup(html, "lxml")

In [25]:
link = soup.find('a')

link['href']      

'https://example.com'

In [26]:
# OR
link.get('href')     # safer

'https://example.com'

### Example: Extracting image sources

`<img src="image.jpg" alt="My Image">`

In [27]:
from bs4 import BeautifulSoup

html = """
<img src="image.jpg" alt="My Image">
"""

soup = BeautifulSoup(html, "lxml")

In [28]:
img = soup.find('img')

img['src']         # image.jpg

'image.jpg'

In [29]:
img.get('alt')     # My Image

'My Image'

### Extracting Lists

In [30]:
from bs4 import BeautifulSoup

html = """
    <ul>
      <li>Python</li>
      <li>Django</li>
      <li>BeautifulSoup</li>
    </ul>
"""

soup = BeautifulSoup(html, "lxml")


In [31]:
items = soup.find_all('li')

for item in items:
    print(item.text)

Python
Django
BeautifulSoup


### Extracting Tables
HTML tables are structured with `<table>`, `<tr>` (table row), `<th>` (header), and `<td>` (cell).

In [32]:
from bs4 import BeautifulSoup

html = """
    <table>
      <tr><th>Name</th><th>Age</th></tr>
      <tr><td>Alice</td><td>25</td></tr>
      <tr><td>Bob</td><td>30</td></tr>
    </table>
"""

soup = BeautifulSoup(html, "lxml")

In [33]:
table = soup.find('table')
rows = table.find_all('tr')

for row in rows:
    cells = row.find_all(['th', 'td'])
    data = [cell.text for cell in cells]
    print(data)


['Name', 'Age']
['Alice', '25']
['Bob', '30']


### Extracting Data with Classes and IDs

In [34]:
from bs4 import BeautifulSoup

html = """
    <div id="profile">
      <p class="name">Alice</p>
      <p class="age">25</p>
    </div>
"""

soup = BeautifulSoup(html, "lxml")


In [35]:
name = soup.find('p', class_='name').text
name

'Alice'

In [36]:
age = soup.find('p', class_='age').text
age

'25'

### Extracting Nested Tags

In [37]:
from bs4 import BeautifulSoup

html = """
    <div class="post">
      <h2>Post Title</h2>
      <p>This is the post content.</p>
    </div>
"""

soup = BeautifulSoup(html, "lxml")

In [39]:
post = soup.find('div', class_='post')
title = post.find('h2').text
content = post.find('p').text

In [42]:
post

<div class="post">
<h2>Post Title</h2>
<p>This is the post content.</p>
</div>

In [40]:
title

'Post Title'

In [41]:
content

'This is the post content.'

### Extracting All Links from a Page

In [44]:
from bs4 import BeautifulSoup

html = """
    <a href="https://example.com">Example</a>
    <a href="/about">About</a>
    <a>Empty link</a>
"""

soup = BeautifulSoup(html, "lxml")


In [45]:
links = soup.find_all('a')

for link in links:
    href = link.get('href')
    if href:
        print(href)


https://example.com
/about


## Real-World Practice: Scrape Quotes and Authors

In [ ]:
import requests
from bs4 import BeautifulSoup

url = "https://quotes.toscrape.com"
response = requests.get(url)
soup = BeautifulSoup(response.text, 'lxml')

quotes = soup.find_all("div", class_="quote")

for quote in quotes:
    text = quote.find("span", class_="text").text
    author = quote.find("small", class_="author").text
    tags = [tag.text for tag in quote.find_all("a", class_="tag")]
    
    print(f"Quote: {text}")
    print(f"Author: {author}")
    print(f"Tags: {tags}")
    print("--------")
